In [ ]:
# Instalación automática (solo la primera vez)
!pip install -q prettytable

import datetime
import csv
import json
import sys
from datetime import timedelta
from io import StringIO
from prettytable import PrettyTable

# Intentar montar Google Drive (silenciosamente si ya está montado)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_MONTADO = True
except:
    DRIVE_MONTADO = False

# ------------------------------------------------------------
# 1. CLASE OBRA
# ------------------------------------------------------------
class Obra:
    def __init__(self, nombre, ubicacion, m2, subsuelos, trabajadores, maquinaria_pesada,
                 obra_nueva, etapa, fecha_inicio, tipo_edificio="residencial", patrimonio_protegido=False):
        self.nombre = nombre
        self.ubicacion = ubicacion.upper()
        self.m2 = m2
        self.subsuelos = subsuelos
        self.trabajadores = trabajadores
        self.maquinaria_pesada = maquinaria_pesada
        self.obra_nueva = obra_nueva
        self.etapa = etapa.lower()
        self.fecha_inicio = fecha_inicio
        self.dias_transcurridos = (datetime.datetime.now() - fecha_inicio).days
        self.tipo_edificio = tipo_edificio
        self.patrimonio_protegido = patrimonio_protegido
        self.documentos_entregados = []

    def registrar_entrega(self, documento, dias_validez=None):
        vencimiento = None
        if dias_validez:
            vencimiento = datetime.datetime.now() + timedelta(days=dias_validez)
        self.documentos_entregados.append({
            "documento": documento,
            "fecha_vencimiento": vencimiento
        })

# ------------------------------------------------------------
# 2. REGLA NORMATIVA
# ------------------------------------------------------------
class ReglaNormativa:
    def __init__(self, condicion_func, documento, alerta, etapa_requerida, plazo_validez_dias=365):
        self.condicion = condicion_func
        self.documento = documento
        self.alerta = alerta
        self.etapa_requerida = etapa_requerida.lower()
        self.plazo_validez = plazo_validez_dias

# ------------------------------------------------------------
# 3. SISTEMA EXPERTO
# ------------------------------------------------------------
class SistemaExpertoLegalAvanzado:
    def __init__(self):
        self.reglas_generales = []
        self.normativas_por_ubicacion = {}
        self.jerarquia_etapas = {"inicio":1, "excavación":2, "construcción":3}

    def agregar_regla_general(self, regla):
        self.reglas_generales.append(regla)

    def cargar_normativas_por_lugar(self, lugar, reglas):
        self.normativas_por_ubicacion[lugar.upper()] = reglas

    def cargar_desde_csv(self, ruta_csv):
        with open(ruta_csv, newline='', encoding='utf-8') as f:
            self._procesar_csv(f)

    def cargar_desde_csv_string(self, csv_string):
        self._procesar_csv(StringIO(csv_string))

    def _procesar_csv(self, file_iterable):
        reader = csv.DictReader(file_iterable)
        for row in reader:
            ubicacion = row['ubicacion'].strip().upper()
            doc = row['documento'].strip()
            alerta = row['alerta'].strip()
            etapa = row['etapa'].strip().lower()
            plazo = int(row['plazo_validez']) if row['plazo_validez'] else 365

            campo = row['condicion'].strip()
            operador = row['operador'].strip()
            valor = row['valor_condicion'].strip()

            if campo == '' or operador == '':
                cond_fn = lambda o: True
            else:
                if operador == '>':
                    if valor.replace('.','',1).isdigit():
                        umbral = float(valor)
                        cond_fn = lambda o, campo=campo, umbral=umbral: getattr(o, campo, 0) > umbral
                    else:
                        cond_fn = lambda o: False
                elif operador == '==':
                    if valor.lower() == 'true':
                        valor_bool = True
                        cond_fn = lambda o, campo=campo, v=valor_bool: getattr(o, campo, False) == v
                    elif valor.lower() == 'false':
                        valor_bool = False
                        cond_fn = lambda o, campo=campo, v=valor_bool: getattr(o, campo, True) == v
                    else:
                        cond_fn = lambda o, campo=campo, v=valor: getattr(o, campo, '') == v
                else:
                    cond_fn = lambda o: False

            regla = ReglaNormativa(cond_fn, doc, alerta, etapa, plazo)
            if ubicacion == 'TODAS':
                self.agregar_regla_general(regla)
            else:
                if ubicacion not in self.normativas_por_ubicacion:
                    self.normativas_por_ubicacion[ubicacion] = []
                self.normativas_por_ubicacion[ubicacion].append(regla)

    def obtener_reglas_aplicables(self, obra):
        reglas = [r for r in self.reglas_generales if r.condicion(obra)]
        if obra.ubicacion in self.normativas_por_ubicacion:
            for r in self.normativas_por_ubicacion[obra.ubicacion]:
                if r.condicion(obra):
                    reglas.append(r)
        return reglas

    def verificar_documentos_faltantes(self, obra):
        nivel_actual = self.jerarquia_etapas.get(obra.etapa, 0)
        reglas = self.obtener_reglas_aplicables(obra)
        faltantes = []
        entregados_nombres = {d['documento'] for d in obra.documentos_entregados}
        for regla in reglas:
            nivel_regla = self.jerarquia_etapas.get(regla.etapa_requerida, 0)
            if nivel_actual >= nivel_regla and regla.documento not in entregados_nombres:
                fecha_vencimiento = obra.fecha_inicio + timedelta(days=regla.plazo_validez)
                dias_restantes = (fecha_vencimiento - datetime.datetime.now()).days
                faltantes.append({
                    "documento": regla.documento,
                    "alerta": regla.alerta,
                    "etapa_requerida": regla.etapa_requerida,
                    "dias_restantes": max(0, dias_restantes),
                    "accion": f"Presentar antes del {fecha_vencimiento.strftime('%d/%m/%Y')}"
                })
        return faltantes

    def verificar_vencimientos(self, obra):
        hoy = datetime.datetime.now()
        alertas = []
        for doc in obra.documentos_entregados:
            if doc['fecha_vencimiento'] is not None:
                dias = (doc['fecha_vencimiento'] - hoy).days
                if dias <= 30:
                    alertas.append({
                        "documento": doc['documento'],
                        "dias_restantes": max(0, dias),
                        "fecha_vencimiento": doc['fecha_vencimiento'].strftime('%d/%m/%Y'),
                        "estado": "VENCIDO" if dias < 0 else "POR VENCER"
                    })
        return alertas

# ------------------------------------------------------------
# DATOS EMBEBIDOS (por si no se encuentran los archivos)
# ------------------------------------------------------------
CSV_EMBEBIDO = """ubicacion,condicion,valor_condicion,operador,documento,alerta,etapa,plazo_validez
TODAS,,,,"Plano aprobado",Critico,inicio,30
TODAS,,,,"Contrato ART",Critico,inicio,365
CABA,m2,1000,>,Permiso municipal,Critico,inicio,180
CABA,patrimonio_protegido,True,==,Permiso patrimonio histórico,Critico,inicio,90
GBA,ubicacion,GBA,==,Permiso GBA,Moderado,inicio,365
GBA,subsuelos,0,>,Estudio de suelos,Moderado,excavación,180
"""

JSON_EMBEBIDO = """
{
    "nombre": "Edificio Torres del Sur",
    "ubicacion": "CABA",
    "m2": 1200,
    "subsuelos": 2,
    "trabajadores": 10,
    "maquinaria_pesada": true,
    "obra_nueva": true,
    "etapa": "excavación",
    "dias_desde_inicio": 10,
    "tipo_edificio": "residencial",
    "patrimonio_protegido": true
}
"""

# ------------------------------------------------------------
# INICIALIZAR SISTEMA Y CARGA DE DATOS
# ------------------------------------------------------------
sistema = SistemaExpertoLegalAvanzado()

# Intentar cargar desde archivos reales (si están en el sistema de archivos de Colab o Drive)
ruta_csv = '/content/drive/MyDrive/TP-Grupal-Jueves/normativas.csv'
ruta_json = '/content/drive/MyDrive/TP-Grupal-Jueves/proyecto_obra.json'

# Ajustá si pusiste los archivos sueltos en /content/
if not DRIVE_MONTADO:
    ruta_csv = 'normativas.csv'
    ruta_json = 'proyecto_obra.json'

cargado_archivos = False
try:
    if not os.path.exists(ruta_csv) or not os.path.exists(ruta_json):
        raise FileNotFoundError
    sistema.cargar_desde_csv(ruta_csv)
    with open(ruta_json, 'r', encoding='utf-8') as f:
        datos_obra = json.load(f)
    fecha_inicio = datetime.datetime.now() - timedelta(days=datos_obra.pop('dias_desde_inicio'))
    obra = Obra(**datos_obra, fecha_inicio=fecha_inicio)
    cargado_archivos = True
    print("✅ Datos cargados desde archivos externos (Drive o entorno local).")
except (FileNotFoundError, Exception) as e:
    print("⚠️  No se encontraron los archivos. Usando datos embebidos para la demo.")
    sistema.cargar_desde_csv_string(CSV_EMBEBIDO)
    datos_obra = json.loads(JSON_EMBEBIDO)
    fecha_inicio = datetime.datetime.now() - timedelta(days=datos_obra.pop('dias_desde_inicio'))
    obra = Obra(**datos_obra, fecha_inicio=fecha_inicio)

# Registramos entregas (con vencimiento)
obra.registrar_entrega("Plano aprobado", dias_validez=30)
obra.registrar_entrega("Contrato ART", dias_validez=365)

# Evaluación
faltantes = sistema.verificar_documentos_faltantes(obra)
vencimientos = sistema.verificar_vencimientos(obra)

# ------------------------------------------------------------
# REPORTE
# ------------------------------------------------------------
print("\n" + "="*70)
print("   SISTEMA EXPERTO LEGAL - CUMPLIMIENTO NORMATIVO DE OBRA")
print("="*70)
print(f"   Obra: {obra.nombre}")
print(f"   Ubicación: {obra.ubicacion}")
print(f"   Etapa actual: {obra.etapa.capitalize()}")
print(f"   Días desde el inicio: {obra.dias_transcurridos}")
print("="*70)

if faltantes:
    print("\n[1] DOCUMENTACIÓN FALTANTE (obligatoria)")
    tabla_falt = PrettyTable()
    tabla_falt.field_names = ["Documento", "Alerta", "Etapa requerida", "Días restantes", "Acción"]
    for doc in faltantes:
        tabla_falt.add_row([doc['documento'], doc['alerta'], doc['etapa_requerida'], doc['dias_restantes'], doc['accion']])
    print(tabla_falt)
else:
    print("\n✅ No hay documentación obligatoria faltante para la etapa actual.")

if vencimientos:
    print("\n[2] VENCIMIENTOS DE DOCUMENTOS ENTREGADOS")
    tabla_venc = PrettyTable()
    tabla_venc.field_names = ["Documento", "Estado", "Días restantes", "Fecha de vencimiento"]
    for v in vencimientos:
        tabla_venc.add_row([v['documento'], v['estado'], v['dias_restantes'], v['fecha_vencimiento']])
    print(tabla_venc)
else:
    print("\n✅ Todos los documentos entregados están vigentes.")

print("\n" + "="*70)
print("RECOMENDACIÓN LEGAL AUTOMÁTICA")
criticos_faltantes = any(d['alerta'] == 'Crítico' and d['dias_restantes'] == 0 for d in faltantes)
criticos_vencidos = any(v['estado'] == 'VENCIDO' for v in vencimientos)
if criticos_faltantes or criticos_vencidos:
    print("⛔ ALERTA MÁXIMA: Documentos críticos vencidos o sin presentar.")
elif any(d['dias_restantes'] < 30 for d in faltantes) or any(v['dias_restantes'] <= 15 for v in vencimientos):
    print("⚠️  PRECAUCIÓN: Plazos ajustados. Gestione con urgencia.")
else:
    print("✅ La obra cumple con la normativa. Puede continuar.")
print("="*70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Datos cargados desde archivos externos (Drive o entorno local).

   SISTEMA EXPERTO LEGAL - CUMPLIMIENTO NORMATIVO DE OBRA
   Obra: Edificio Torres del Sur
   Ubicación: CABA
   Etapa actual: Excavación
   Días desde el inicio: 10

[1] DOCUMENTACIÓN FALTANTE (obligatoria)
+------------------------------+---------+-----------------+----------------+--------------------------------+
|          Documento           |  Alerta | Etapa requerida | Días restantes |             Acción             |
+------------------------------+---------+-----------------+----------------+--------------------------------+
|      Permiso municipal       | Critico |      inicio     |      169       | Presentar antes del 05/11/2026 |
| Permiso patrimonio histórico | Critico |      inicio     |       79       | Presentar antes del 07/08/2026 |
+------------------------------+---------